# 05c Phased schedule

The phased design in 02 moves the 20 nodes six times after the reconnaissance grid. Every move is a field trip, so
this notebook asks how much imaging is lost by moving them only **two to four** times, which layout makes the
best use of those few moves, and how long each deployment should stay out.

Every configuration starts with the same wide phase 1 grid from 02 (`designs.json`, 5 x 4 nodes over 8 x 6 km).
The BDT high is picked from the phase 1 image, then the nodes are moved K times (K = 2, 3, 4), with each phase 2
deployment placed around that pick. Phase 2 layouts are built from the 02 station bank (0.6 km lattice), so the
03 travel-time tables already cover them.

| Layout | Phase 2 deployments |
|---|---|
| `rolling` | K contiguous 5 x 4 patches at 0.6 km that tile one block around the pick (02's scheme, with fewer patches). Short apertures within each deployment. |
| `interleaved` | The same block and the same sites as `rolling`, but each deployment takes every K-th site, so every deployment spans the whole block at a wider spacing. |
| `zoom` | First move: a 5 x 4 grid at 1.2 km centred on the pick. The BDT high is re-picked from everything recorded so far, then the remaining K - 1 moves are rolling 0.6 km patches around the new pick. |

**Why noise is on.** With noise-free echoes, recording time does not change the image at all (05 only uses
"did the pair record together"). Deployment length only matters through noise, so this notebook uses the noise
model of 06: each pair's correlation noise has std `NSR x SIGNAL x sqrt(DAYS_REF / days)`. Two changes from 06
make time matter properly:

- **Picks only use data recorded by then.** The phase 1 pick (and the zoom re-pick) is made from the data
  recorded up to that day, including the Trillium and Mercury pairs, not from three years of permanent data.
- **One noise record per station pair.** Noise is stacked from a fixed weekly record per pair, so the data
  behind a pick is part of the final image, and schedules are compared on the same noise.

**Earth:** the `dome` BDT at a few conductor positions, with **Vp 5 % higher** inside the BDT zone (the
quartz-cemented case in `docs/03_rock_physics_notes.md`). Under the Born approximation the image scales with the
contrast, so +5 % and -5 % give almost the same |image|. This is really a test of a 5 % contrast.

**Two tests:**

- **A, layout:** every layout x K, at one schedule (phase 1 `P1_DAYS_A`, each move `P2_DAYS_A`), plus the 6-move
  rolling design from 02 as the reference.
- **B, deployment length:** one layout, every K, over a grid of phase 1 and phase 2 lengths.

Costs are given as field trips (install + K moves + retrieval, plus any battery-service visits) and node-days.

Same forward model, imaging and scoring as 05/06 (`mi_common.py`).

Inputs: everything 05 reads. Output: `data/results_05c_schedule.csv`

In [ ]:
# --- Imports ---
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
PHASE = "p"
DROP = -0.05                         # perturbation() takes a drop, so -0.05 = Vp 5 % higher inside the BDT zone
SCENARIOS = ["pos00_dome", "pos02_dome", "pos10_dome"]   # dome at the drill site, 2 km off, 4 km off
NSR = 1.0                            # noise / signal for DAYS_REF days of shared recording (09 treats 1.0 as realistic)
DAYS_REF = 30
N_REAL = 2                           # noise realisations per scenario

GEOMETRIES = ["rolling", "interleaved", "zoom"]
N_MOVES = [2, 3, 4]                  # phase 2 deployments (= node moves after phase 1)
REFERENCE = ("rolling", 6)           # 02's design, for comparison

# Test A: layout, at one schedule
P1_DAYS_A, P2_DAYS_A = 84, 42
# Test B: deployment length, one layout (None = best mean coverage in test A)
LENGTH_GEOMETRY = None
P1_DAYS_B = [42, 84]
P2_DAYS_B = [28, 42, 84]

# Field cost
NODE_BATTERY_DAYS = None             # battery life of a node (with any external pack). A deployment longer than
                                     # this needs a service visit, counted as an extra trip. None = no limit
N_NODES = 20

PATCH = (5, 4)                       # nodes along x across the node axis in one deployment
FINE, COARSE = 0.6, 1.2              # km; multiples of the 02 bank spacing
RESULTS = "../data/results_05c_schedule.csv"

mi = MatrixImaging(PHASE, DAYS_REF=DAYS_REF)
print(mi)

## Phase 2 layouts

All layouts are built on the 02 station bank: lattice index `i` along the node axis (AZ = 120°) and `j` across
it, 0.6 km apart. The block for `rolling` and `interleaved` is K patches arranged as below (K = 6 is 02's
15 x 8 block):

| K | patches along x across | block (sites) | block size (km) |
|---|---|---|---|
| 1 | 1 x 1 | 5 x 4 | 2.4 x 1.8 |
| 2 | 2 x 1 | 10 x 4 | 5.4 x 1.8 |
| 3 | 3 x 1 | 15 x 4 | 8.4 x 1.8 |
| 4 | 2 x 2 | 10 x 8 | 5.4 x 4.2 |
| 6 | 3 x 2 | 15 x 8 | 8.4 x 4.2 |

In [ ]:
# --- Layouts on the station bank ---
SP, C0, AZR = mi.P2["spacing"], np.array(mi.P2["centre0"], float), mi.AZR
P1_IDS = mi.D["designs"]["phased"][0]["stations"]
BLOCK = {1: (1, 1), 2: (2, 1), 3: (3, 1), 4: (2, 2), 6: (3, 2)}   # patches along x across


def to_axis(xy):
    # Map point -> (along, across) km from the bank centre, kept within P2_SEARCH_R as in 02/05.
    d = np.asarray(xy, float) - C0
    if np.hypot(*d) > mi.P2_SEARCH_R:
        d *= mi.P2_SEARCH_R / np.hypot(*d)
    return np.array([d[0] * np.sin(AZR) + d[1] * np.cos(AZR), -d[0] * np.cos(AZR) + d[1] * np.sin(AZR)])


def lattice(centre, n, step):
    # Bank indices (i along, j across) of an n[0] x n[1] grid at step km, snapped as a whole onto the bank.
    m = int(round(step / SP))
    out = []
    for c, nn in zip(centre, n):
        frac = ((nn - 1) * m / 2) % 1                      # 0, or 0.5 when the grid has no middle site
        base = np.round(c / SP - frac)
        out.append(np.rint(base + frac + (np.arange(nn) - (nn - 1) / 2) * m).astype(int))
    return out


def snake(na, nc):
    return [(a, b) for a in range(na) for b in (range(nc) if a % 2 == 0 else reversed(range(nc)))]


def phase2_sites(geometry, K, centre):
    # Lists of bank (i, j) sites, one list per deployment, for K rolling or interleaved deployments around centre.
    na, nc = BLOCK[K]
    I, J = lattice(centre, (PATCH[0] * na, PATCH[1] * nc), FINE)
    if geometry == "rolling":
        return [[(i, j) for i in I[PATCH[0] * a:PATCH[0] * (a + 1)] for j in J[PATCH[1] * b:PATCH[1] * (b + 1)]]
                for a, b in snake(na, nc)]
    if geometry == "interleaved":
        cls = (lambda a, b: (a + b) % K) if nc == 1 else (lambda a, b: (a % na) * nc + b % nc)
        return [[(I[a], J[b]) for a in range(len(I)) for b in range(len(J)) if cls(a, b) == q] for q in range(K)]
    raise ValueError(geometry)


def zoom_first(centre):
    I, J = lattice(centre, PATCH, COARSE)
    return [(i, j) for i in I for j in J]


def ids(sites):
    # Station ids of bank sites (sites in water or off the bank are dropped, as in 05).
    return [mi.BANK[s] for s in sites if s in mi.BANK]


# every layout gives N_NODES distinct sites per deployment
for g in ("rolling", "interleaved"):
    for K in N_MOVES + [REFERENCE[1]]:
        dep = phase2_sites(g, K, (0.0, 0.0))
        assert all(len(set(d)) == N_NODES for d in dep) and len({s for d in dep for s in d}) == N_NODES * K, (g, K)
assert len(set(zoom_first((0.0, 0.0)))) == N_NODES

In [ ]:
# --- Map of each layout, for a BDT high picked at the bank centre (the zoom re-pick assumed to land there too) ---
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]   # deployment 1, 2, ... in order
ST = mi.stations


def example(geometry, K):
    if geometry == "zoom":
        return [zoom_first((0, 0))] + (phase2_sites("rolling", K - 1, (0, 0)) if K > 1 else [])
    return phase2_sites(geometry, K, (0, 0))


with plt.style.context("default"):
    fig, axs = plt.subplots(len(GEOMETRIES), len(N_MOVES), figsize=(3.3 * len(N_MOVES), 3.3 * len(GEOMETRIES)),
                            sharex=True, sharey=True)
    for r, g in enumerate(GEOMETRIES):
        for c, K in enumerate(N_MOVES):
            ax = axs[r, c]
            ax.plot(ST.x[mi.PERMANENT], ST.y[mi.PERMANENT], "^", color="#8a8984", ms=4, label="Trillium / Mercury")
            ax.plot(ST.x[P1_IDS], ST.y[P1_IDS], "o", mfc="none", mec="k", ms=5, label="Phase 1")
            for q, sites in enumerate(example(g, K)):
                s = ids(sites)
                ax.plot(ST.x[s], ST.y[s], "o", color=COL[q], ms=3.5, mec="white", mew=0.5, label=f"Move {q + 1}")
            ax.set(aspect="equal", xlim=(-8, 8), ylim=(-8, 8), title=f"{g}, {K} moves")
            if r == len(GEOMETRIES) - 1:
                ax.set_xlabel("East (km)")
            if c == 0:
                ax.set_ylabel("North (km)")
    h, l = axs[-1, -1].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()

## Noise, recording time and imaging

The noise on pair (i, j) is the stack of its daily correlations. Each pair has one fixed noise record per
realisation: a unit draw per week (`BLOCK_DAYS`), seeded by (realisation, pair) alone. A pair's noise is the sum of
the weeks it recorded, weighted by the days shared in each week, divided by its total shared days. So:

- a pick and the final image of one run see the same noise on the data they share;
- two schedules that share recording days share the same noise (common random numbers). Differences between
  schedules come from the schedules themselves, not from a lucky draw.

Permanent stations record from day 0. The final image uses the whole Trillium deployment (`DEPLOY_END` days), as
in 05/06.

In [ ]:
# --- Reference signal (as in 06): rms echo of the central dome recorded by the grid design, noise-free ---
NF = len(mi.F)
ref_name = SCENARIOS[0]
_ids = sorted(set(mi.PERMANENT) | set(mi.D["designs"]["grid"][0]["stations"]))
_k, _sig = mi.perturbation(ref_name, DROP)
_R = mi.forward(_ids, _k, _sig)
SIGNAL = float(np.sqrt(np.mean(np.abs(_R[:, mi.pair_mask(_ids, mi.windows(mi.D["designs"]["grid"]))]) ** 2)))
LEVEL = NSR * SIGNAL
del _R
print(f"reference {ref_name}: rms signal {SIGNAL:.2e}, noise level {LEVEL:.2e} for {DAYS_REF} days shared")

In [ ]:
# --- Recording windows, echoes and noise ---
def windows(phases, t0, t1):
    # Recording windows clipped to [t0, t1]: permanent stations throughout, nodes during their phase.
    w = {s: [(t0, t1)] for s in mi.PERMANENT}
    for p in phases:
        a, b = max(p["start"], t0), min(p["end"], t1)
        for s in p["stations"]:
            w.setdefault(s, [])
            if b > a:
                w[s].append((a, b))
    return w


class Echoes:
    # Noise-free reflection matrix for one scenario. Station pairs are computed once and kept, so a pick and the
    # final image, and the configurations that share stations, only pay for pairs not seen before.

    def __init__(self, name):
        self.k, self.sig = mi.perturbation(name, DROP)
        self.pos = {}
        self.R = np.zeros((NF, 64, 64), np.complex64)
        self.known = np.zeros((64, 64), bool)

    def _grow(self, n):
        if n > len(self.known):
            m = max(n, 2 * len(self.known))
            R = np.zeros((NF, m, m), np.complex64)
            known = np.zeros((m, m), bool)
            c = len(self.known)
            R[:, :c, :c], known[:c, :c] = self.R, self.known
            self.R, self.known = R, known

    def get(self, ids):
        for s in ids:
            self.pos.setdefault(s, len(self.pos))
        self._grow(len(self.pos))
        sel = np.array([self.pos[s] for s in ids])
        todo = ~self.known[np.ix_(sel, sel)].all(axis=1)          # stations with at least one pair not computed
        if todo.any():
            k = self.k
            T = np.stack([mi.station_tt(s)[0][k] for s in ids]).astype(np.float64)
            A = np.stack([mi.spreading(s, mi.tx[mi.SI[k]], mi.ty[mi.SJ[k]], mi.tz[mi.SK[k]]) for s in ids])
            G = A * np.exp(-2j * np.pi * mi.F[0] * T)                # frequency by frequency, by phase steps
            step = np.exp(-2j * np.pi * (mi.F[1] - mi.F[0]) * T) if NF > 1 else None
            rows = sel[todo]
            for n in range(NF):
                blk = ((G[todo] * self.sig) @ G.T).astype(np.complex64)
                self.R[n][np.ix_(rows, sel)] = blk
                self.R[n][np.ix_(sel, rows)] = blk.T
                if n < NF - 1:
                    G *= step
            self.known[np.ix_(rows, sel)] = self.known[np.ix_(sel, rows)] = True
        return self.R[:, sel][:, :, sel]


BLOCK_DAYS = 7                                         # noise record resolution (days)
EDGES = np.arange(int(np.ceil(mi.DEPLOY_END / BLOCK_DAYS)) + 1) * BLOCK_DAYS


def shared(w1, w2):
    return [(max(a1, a2), min(b1, b2)) for a1, b1 in w1 for a2, b2 in w2 if min(b1, b2) > max(a1, a2)]


def noisy(R, ids, phases, t_now, real):
    # R plus stacked correlation noise for data recorded up to t_now. Returns (R + noise, shared days).
    win = windows(phases, 0, t_now)
    days = mi.overlap_days(ids, win)
    if not LEVEL:
        return R, days
    lo, hi = EDGES[:-1], EDGES[1:]
    acc = np.zeros(R.shape, np.complex64)
    for p, q in zip(*np.nonzero(np.triu(days > 0, 1))):
        d = sum(np.clip(np.minimum(hi, b) - np.maximum(lo, a), 0, None) for a, b in shared(win[ids[p]], win[ids[q]]))
        nz = np.nonzero(d)[0]
        # the pair's whole weekly noise record, drawn from (realisation, pair) alone
        Z = np.random.default_rng([real, *sorted((int(ids[p]), int(ids[q])))]).standard_normal(
            (len(lo), 2, NF), dtype=np.float32)
        acc[:, p, q] = ((Z[nz, 0] + 1j * Z[nz, 1]) * np.sqrt(d[nz] / 2)[:, None]).sum(axis=0)
    acc += acc.transpose(0, 2, 1)
    std = LEVEL * np.sqrt(DAYS_REF) / np.maximum(days, 1e-9) * (days > 0)
    return (R + acc * std).astype(np.complex64), days


def image_at(echo, phases, t_now, real):
    # Image from every station that has recorded by t_now, with the noise of the data recorded by then.
    ids = sorted(set(mi.PERMANENT) | {s for p in phases if p["start"] < t_now for s in p["stations"]})
    Rn, days = noisy(echo.get(ids), ids, phases, t_now, real)
    return mi.image(ids, Rn, days > 0), ids, days


def pick(echo, name, phases, t_now, real):
    # BDT high picked from the data recorded by t_now; the bank centre if nothing is imaged. (xy, error km)
    s = mi.score(image_at(echo, phases, t_now, real)[0], name)
    ok = np.isfinite(s["crest_x"])
    return (np.array([s["crest_x"], s["crest_y"]]) if ok else C0), s["crest_err_km"]

In [ ]:
# --- One configuration ---
def trips(lengths):
    # Install + one per move + retrieval, plus service visits for deployments longer than the battery lasts.
    extra = 0 if NODE_BATTERY_DAYS is None else sum(int(np.ceil(L / NODE_BATTERY_DAYS)) - 1 for L in lengths)
    return len(lengths) + 1 + extra


_p1_cache = {}


def run_config(echo, name, real, geometry, K, p1_days, p2_days):
    p1 = dict(start=0, end=p1_days, stations=P1_IDS)
    key = (name, real, p1_days)
    if key not in _p1_cache:                                   # the phase 1 pick is shared by every layout
        _p1_cache[key] = pick(echo, name, [p1], p1_days, real)
    xy1, err1 = _p1_cache[key]
    phases, t, extra = [p1], p1_days, dict(p1_error_km=err1)

    if geometry == "zoom":
        phases.append(dict(start=t, end=t + p2_days, stations=ids(zoom_first(to_axis(xy1)))))
        t += p2_days
        xy2, err2 = pick(echo, name, phases, t, real)
        extra["zoom_error_km"] = err2
        rest = phase2_sites("rolling", K - 1, to_axis(xy2)) if K > 1 else []
    else:
        rest = phase2_sites(geometry, K, to_axis(xy1))
    for sites in rest:
        phases.append(dict(start=t, end=t + p2_days, stations=ids(sites)))
        t += p2_days

    img, all_ids, days = image_at(echo, phases, mi.DEPLOY_END, real)
    lengths = [p["end"] - p["start"] for p in phases]
    return dict(scenario=name, real=real, geometry=geometry, n_moves=K, p1_days=p1_days, p2_days=p2_days, nsr=NSR,
                campaign_days=t, trips=trips(lengths), node_days=N_NODES * sum(lengths),
                n_stations=len(all_ids), n_pairs=int((days > 0).sum() // 2), **mi.score(img, name), **extra)


KEYS = ["scenario", "real", "geometry", "n_moves", "p1_days", "p2_days", "nsr"]


def run_all(configs, results=RESULTS):
    # configs: (geometry, K, p1_days, p2_days). Every scenario x realisation; rerunning skips rows already saved.
    done = pd.read_csv(results) if os.path.exists(results) else pd.DataFrame(columns=KEYS)
    have = set(map(tuple, done[KEYS].astype(str).values))
    for name in SCENARIOS:
        echo = None
        for real in range(N_REAL):
            for g, K, a, b in configs:
                if tuple(map(str, (name, real, g, K, a, b, NSR))) in have:
                    continue
                echo = echo or Echoes(name)
                t0 = time.time()
                row = run_config(echo, name, real, g, K, a, b)
                done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
                done.to_csv(results, index=False)
                print(f"{name} r{real} {g:<11} K={K} {a}+{K}x{b} d: coverage {row['coverage']:.2f} "
                      f"({time.time() - t0:.0f} s)", flush=True)
    m = done.nsr.astype(float) == NSR
    want = {(g, int(K), int(a), int(b)) for g, K, a, b in configs}
    sel = [(g, int(K), int(a), int(b)) in want for g, K, a, b in done[["geometry", "n_moves", "p1_days", "p2_days"]]
           .itertuples(index=False)]
    return done[m & np.array(sel, bool) & done.scenario.isin(SCENARIOS)].astype(
        dict(n_moves=int, p1_days=int, p2_days=int, real=int))

## Test A: layout

Every layout x number of moves at one schedule, plus the 6-move reference.

In [ ]:
CONFIGS_A = [(g, K, P1_DAYS_A, P2_DAYS_A) for g in GEOMETRIES for K in N_MOVES] + \
            [(*REFERENCE, P1_DAYS_A, P2_DAYS_A)]
resA = run_all(CONFIGS_A)

In [ ]:
# --- Summary per layout and number of moves (over scenarios and noise realisations) ---
AGG = dict(coverage_mean=("coverage", "mean"), coverage_worst=("coverage", "min"),
           crest_err_km=("crest_err_km", "median"), top_err_km=("top_err_km", "median"),
           p1_error_km=("p1_error_km", "median"), trips=("trips", "first"), campaign_days=("campaign_days", "first"),
           node_days=("node_days", "first"))
sumA = resA.groupby(["geometry", "n_moves"]).agg(**AGG)
sumA.round(2)

In [ ]:
# --- Coverage and BDT-high error against the number of moves ---
ref = sumA.loc[REFERENCE]
GCOL = dict(zip(GEOMETRIES, ["#2a78d6", "#eb6834", "#1baf7a"]))
panels = [("coverage_mean", "Mean coverage of BDT top"), ("coverage_worst", "Worst-case coverage"),
          ("crest_err_km", "BDT high error, median (km)")]

with plt.style.context("default"):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    for ax, (col, lab) in zip(axs, panels):
        for g in GEOMETRIES:
            s = sumA.loc[g].reindex(N_MOVES)[col]
            ax.plot(s.index, s.values, "o-", color=GCOL[g], lw=2, ms=8, mec="white", label=g)
        ax.axhline(ref[col], color="#52514e", ls="--", lw=1, label=f"{REFERENCE[1]}-move reference")
        ax.set(xticks=N_MOVES, xlabel="Phase 2 moves (K)", ylabel=lab)
        if col.startswith("coverage"):
            ax.set_ylim(0, 1.02)
        ax.grid(axis="y", color="#e3e2dd", lw=0.6)
        ax.spines[["top", "right"]].set_visible(False)
    axs[0].legend(frameon=False, fontsize=8)
    fig.suptitle(f"Test A: phase 1 {P1_DAYS_A} d, then K moves of {P2_DAYS_A} d each (NSR {NSR})", fontsize=10)
    plt.tight_layout()
    plt.show()

## Test B: deployment length

One layout (the best in test A unless `LENGTH_GEOMETRY` is set), every K, over the grid of phase 1 and phase 2
lengths. The phase 1 length sets how well the BDT high is found before the nodes move; the phase 2 length sets
the noise of the dense data.

In [ ]:
GEOM_B = LENGTH_GEOMETRY or sumA.drop(REFERENCE).coverage_mean.groupby("geometry").mean().idxmax()
print("test B layout:", GEOM_B)
CONFIGS_B = [(GEOM_B, K, a, b) for K in N_MOVES for a in P1_DAYS_B for b in P2_DAYS_B]
resB = run_all(CONFIGS_B)
sumB = resB.groupby(["n_moves", "p1_days", "p2_days"]).agg(**AGG)
sumB.round(2)

In [ ]:
# --- Mean coverage over phase 1 x phase 2 length, one panel per K ---
with plt.style.context("default"):
    fig, axs = plt.subplots(1, len(N_MOVES), figsize=(3.6 * len(N_MOVES) + 1, 3.2), sharey=True)
    for ax, K in zip(np.atleast_1d(axs), N_MOVES):
        t = sumB.loc[K].coverage_mean.unstack("p2_days").reindex(index=P1_DAYS_B, columns=P2_DAYS_B)
        im = ax.imshow(t.values, cmap="Blues", vmin=0, vmax=1, origin="lower", aspect="auto")
        for i in range(t.shape[0]):
            for j in range(t.shape[1]):
                v = t.values[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if v > 0.5 else "#0b0b0b")
        ax.set(xticks=range(len(P2_DAYS_B)), xticklabels=P2_DAYS_B, yticks=range(len(P1_DAYS_B)),
               yticklabels=P1_DAYS_B, xlabel="Each phase 2 deployment (days)", title=f"{GEOM_B}, {K} moves")
    np.atleast_1d(axs)[0].set_ylabel("Phase 1 (days)")
    fig.colorbar(im, ax=axs, label="Mean coverage of BDT top", shrink=0.9)
    plt.show()

In [ ]:
# --- What each schedule buys: coverage against campaign length, by number of moves ---
MK = {2: "o", 3: "s", 4: "D"}
KCOL = dict(zip(N_MOVES, ["#2a78d6", "#eb6834", "#1baf7a"]))
s = sumB.reset_index()

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 4))
    for K in N_MOVES:
        g = s[s.n_moves == K]
        for a, fill in zip(P1_DAYS_B, ["white", None]):
            h = g[g.p1_days == a]
            ax.plot(h.campaign_days, h.coverage_mean, MK.get(K, "o"), ms=9, mec=KCOL[K], mew=2,
                    mfc=fill or KCOL[K], label=f"{K} moves ({K + 2} trips), phase 1 {a} d")
    ax.axhline(ref.coverage_mean, color="#52514e", ls="--", lw=1,
               label=f"{REFERENCE[1]}-move reference ({int(ref.trips)} trips, {int(ref.campaign_days)} d)")
    ax.set(xlabel="Node campaign length (days)", ylabel="Mean coverage of BDT top", ylim=(0, 1.02))
    ax.grid(color="#e3e2dd", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
# --- All schedules tested, cheapest first: field trips, campaign length, node-days and scores ---
allres = pd.concat([resA, resB]).drop_duplicates(KEYS)
table = (allres.groupby(["geometry", "n_moves", "p1_days", "p2_days"]).agg(**AGG)
         .reset_index().sort_values(["trips", "campaign_days", "coverage_mean"], ascending=[True, True, False]))
print(table.round(2).to_string(index=False))

## Reading the results

- **Moves:** compare each K with the dashed 6-move line in test A. If 3 moves are within the noise-realisation
  spread of 6, the extra trips are not buying imaging.
- **Layout:** `interleaved` puts long and short offsets in every deployment, `rolling` gives dense contiguous
  coverage but each deployment sees a small aperture, and `zoom` spends one move to fix a poor phase 1 pick.
  `zoom` should win when the phase 1 error (`p1_error_km`) is large, i.e. short phase 1 or high noise.
- **Length:** at fixed trips, longer deployments only help while noise limits the image. Look for where the
  test B heatmap flattens. Compare schedules of the same campaign length (e.g. 2 x 63 d against 3 x 42 d) in the
  last plot to see whether a fourth trip beats longer recording.

Caveats: the noise level (`NSR`) is unknown (06 scans it), so rerun with `NSR = 0.5` and `2.0` before choosing.
`NODE_BATTERY_DAYS` should be set to the nodes actually used. Only the dome is tested, at three conductor
positions.